In [ ]:
import geopandas as gpd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from sklearn.metrics import f1_score, precision_score, recall_score, cohen_kappa_score

# Load the mesh shapefile
mesh = gpd.read_file("/home/stagiaire/Téléchargements/PR/D/mesh_rj.shp")

# Lists to store sensitivity results
f1_scores_sensib = []
precision_scores_sensib = []
recall_scores_sensib = []
kappa_scores_sensib = []

# Loop over different threshold values
for p in [i / 10 for i in range(1, 10)]:
    
    # Define labels based on conditions
    mesh['label'] = np.where(
        (mesh['vegetation'] <= 0.95) & 
        (mesh['ghsl'] >= 0.5) & 
        (mesh['osm'] <= 0.5) & 
        (mesh['favelas'] > p),
        1, 
        np.where(
            (mesh['vegetation'] <= 0.95) & 
            (mesh['ghsl'] >= 0.5) & 
            (mesh['osm'] <= 0.5) & 
            (mesh['favelas'] == 0),
            0, 
            np.nan 
        )
    )

    # Filter dataset to exclude NaN labels
    dataset = mesh[mesh['label'].notna()]

    # Load the zones shapefile
    zones = gpd.read_file("/home/stagiaire/Téléchargements/PR/D/zones.shp") 

    dataset = dataset.copy()

    # Assign each cell to a zone based on centroid location
    dataset['centroid'] = dataset.geometry.centroid
    points_zones = gpd.sjoin(dataset.set_geometry('centroid'), zones[['fid', 'geometry']], how='left', predicate='within')
    dataset.loc[:, 'zone'] = points_zones['fid']
    dataset = dataset.drop(columns=['centroid'])

    # Remove rows without assigned zones
    dataset = dataset[dataset['zone'].notna()]

    # Get unique zones and reverse order
    zones = dataset['zone'].unique().tolist()
    zones.reverse()

    # Feature columns
    f_cols = ['vegetation', 'slope', 'profile_co', 'entropy', 'nodes', 'roads', 'mean_conne', 'min_connex', 'max_connex'] 

    # Initialize performance metric lists
    f1_scores = [[] for _ in range(len(zones))]
    precision_scores = [[] for _ in range(len(zones))]
    recall_scores = [[] for _ in range(len(zones))]
    kappa_scores = [[] for _ in range(len(zones))]

    # Perform 5-fold cross-validation
    for _ in range(5):
        
        folds = []
        
        for z in zones:
            dataset_zone = dataset[dataset['zone'] == z]
            X, y = dataset_zone[f_cols].values, dataset_zone['label'].values
            
            class_0 = X[y == 0]
            class_1 = X[y == 1]
            
            # Balance the dataset
            if len(class_0) > len(class_1):
                class_0_downsampled = resample(class_0, replace=False, n_samples=len(class_1))
                X_balanced = np.vstack([class_0_downsampled, class_1])
                y_balanced = np.hstack([np.zeros(len(class_0_downsampled)), np.ones(len(class_1))])
            else:
                class_1_downsampled = resample(class_1, replace=False, n_samples=len(class_0))
                X_balanced = np.vstack([class_0, class_1_downsampled])
                y_balanced = np.hstack([np.zeros(len(class_0)), np.ones(len(class_1_downsampled))])
                
            # Shuffle the dataset
            p = np.random.permutation(len(y_balanced))
            
            X_balanced, y_balanced = X_balanced[p], y_balanced[p]
            
            folds.append([X_balanced, y_balanced])

        # Perform cross-validation training and testing
        for i in range(len(folds)):
            X_test, y_test = folds[i][0], folds[i][1]
            
            X_train = np.vstack([fold[0] for j, fold in enumerate(folds) if j != i])
            y_train = np.hstack([fold[1] for j, fold in enumerate(folds) if j != i])
            
            # Train Random Forest classifier
            clf = RandomForestClassifier()
            clf.fit(X_train, y_train)
            
            # Predict labels
            y_pred = clf.predict(X_test)

            # Compute evaluation metrics
            precision_scores[i].append(precision_score(y_test, y_pred))
            recall_scores[i].append(recall_score(y_test, y_pred))
            f1_scores[i].append(f1_score(y_test, y_pred))
            kappa_scores[i].append(cohen_kappa_score(y_test, y_pred))
    
    '''
    for i in range(len(zones)):
        print(f"Precision zone {i+1}: {np.mean(precision_scores[i]):.2f} +/- {np.std(precision_scores[i]):.2f}")
        print(f"Recall zone {i+1}: {np.mean(recall_scores[i]):.2f} +/- {np.std(recall_scores[i]):.2f}")
        print(f"F1-score zone {i+1}: {np.mean(f1_scores[i]):.2f} +/- {np.std(f1_scores[i]):.2f}")
        print(f"Kappa zone {i+1}: {np.mean(kappa_scores[i]):.2f} +/- {np.std(kappa_scores[i]):.2f}\n")
    ''' 

    # Compute overall metrics
    print(f"Precision: {np.mean([np.mean(f) for f in precision_scores]):.2f} +/- {np.std([np.mean(f) for f in precision_scores]):.2f}")
    print(f"Recall: {np.mean([np.mean(f) for f in recall_scores]):.2f} +/- {np.std([np.mean(f) for f in recall_scores]):.2f}")
    print(f"F1-score: {np.mean([np.mean(f) for f in f1_scores]):.2f} +/- {np.std([np.mean(f) for f in f1_scores]):.2f}")
    print(f"Kappa: {np.mean([np.mean(f) for f in kappa_scores]):.2f} +/- {np.std([np.mean(f) for f in kappa_scores]):.2f}")
    
    # Store sensitivity results
    f1_scores_sensib.append(np.mean([np.mean(f) for f in precision_scores]))
    precision_scores_sensib.append(np.mean([np.mean(f) for f in recall_scores]))
    recall_scores_sensib.append(np.mean([np.mean(f) for f in f1_scores]))
    kappa_scores_sensib.append(np.mean([np.mean(f) for f in kappa_scores]))


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager 

font_path = "/usr/share/fonts/truetype/cmu/cmunrm.ttf" 
font_manager.fontManager.addfont(font_path)
font_prop = font_manager.FontProperties(fname=font_path)

plt.rcParams['font.family'] = font_prop.get_name()
plt.tick_params(axis='both', which='major', labelsize=12) 
plt.tick_params(axis='both', which='minor', labelsize=10) 

#plt.plot(range(10, 91, 10), precision_scores_sensib, marker="o", linestyle="-", label = "Precision")
#plt.plot(range(10, 91, 10), recall_scores_sensib, marker="o", linestyle="-", label = "Recall")
plt.plot(range(10, 91, 10), f1_scores_sensib, marker="o", linestyle="-", label = "F1-score")
plt.plot(range(10, 91, 10), kappa_scores_sensib, marker="o", linestyle="-", label = "Kappa")
plt.xlabel("Favela Coverage Proportion(%)", fontsize=12)
plt.ylabel("Performance Metrics", fontsize=12)
plt.legend(loc='lower right', fontsize=12)
plt.grid()
plt.savefig("sensitivity_analysis.png", dpi=400, bbox_inches="tight")
plt.show()